# PGA DraftKings — weekly (API draft)

The weekly routine rebuilt on the PGA Tour's own API. **Nothing to type:** this week's
event comes from the Tour's schedule, last week's results arrive with the refresh, and every
name — DraftKings, golfodds — is matched to the Tour's player id where it enters.

**Draft.** Sections 1–5 run for real. Section 6 (the model) is not ported yet: until it is,
run `pga-dk.ipynb` for predictions, lineups and the dashboard. Nothing here writes
`golf.db`.

Run top to bottom. **At work, skip 4a.**

## 1. Setup

In [ ]:
from IPython import get_ipython

# Autoreload picks up edits to pga_api/ without a restart.
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("load_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

from pga_api import weekly
from pga_api.weekly import find_player, add_alias

## 2. Refresh the data

Rebuilds `data/pga.db` from the Tour's API: results of every finished event (last week's
included, automatically), this season's stats, and the entry lists of events starting in
the next ten days. Seconds when the cache is warm; about five minutes on a new computer.

**Prints** what has finished since the last run. If last week's event is not listed, the
Tour has not marked it final yet — run again later.

In [ ]:
new_events = weekly.refresh()

## 3. This week

The next event to start. When two share the week it takes the one with more of the world's
top 50 in its field — the main event — and names the other. Events whose past editions were
not stroke play (Presidents Cup, Ryder Cup, Zurich) are skipped.

**To use a different event**, pass its id: `weekly.this_week(pick="R2026554")`.
The entry list is usually published the Friday before; until then it reads 0 players,
which is normal.

In [ ]:
week = weekly.this_week()

## 4. DraftKings prices

### 4a. Download · SKIP AT WORK

The only cell that talks to DraftKings (the work network blocks it). Saves the week's file
to `data/salaries/` — **commit it**. Safe to re-run all week; an unchanged file says so.

In [ ]:
from utils import dk_api

dk_api.refresh_from_dk(week.config)

### 4b. Priced field · ALWAYS RUN

Reads the saved file — no network — and matches every DraftKings name to a Tour player id.
**Look for:** `every DraftKings name resolved`. Anything else prints the one line that fixes
it (section 5b).

In [ ]:
dk = weekly.prices(week)
dk.head()

## 5. Odds

### 5a. Scrape and save

Scrapes golfodds.com and **checks the board is this event** — by its golfers against the
entry list, or by its dates before the list is out. A wrong board is refused, not saved.
Saves to `data/odds/` — **commit it**: nobody can fetch last week's board later.

**Run it before the first tee time.** An empty board early in the week is normal; re-run
once it opens.

In [ ]:
odds = weekly.odds(week)

### 5b. Fix a name (only when 4b or 5a asks)

Find the golfer's id, then save the alias. It applies to every source from then on and is
committed in `data/player_aliases.csv`.

In [ ]:
# find_player("Bhatia")
# add_alias("Akshay Bhatia Jr", "56630")

## 6. Model, lineups, dashboard — not ported yet

What goes here, in order, once the features read `pga.db`:

1. **Report card** — last week's logged predictions graded against the results section 2
   just brought in.
2. **Features** — the same point-in-time features as today, keyed by player id, over every
   stroke-play event (opposite-field events and TOUR Championships included).
3. **Train and score** this week's DraftKings field; save the predictions before the first tee.
4. **Export and open the dashboard.**

Until then: `pga-dk.ipynb` from *Model Pipeline* down. Its `golf.db` is repaired and now holds
the same history (September 2026).

---
## Appendix — audit against golf.db

Only needed while `golf.db` is still in use. Compares the two databases and shows every
difference by kind; after the September 2026 repair the counts are near zero.

In [ ]:
from pga_api import compare

r = compare.report()

In [ ]:
r["events"].loc[r["events"]["id_wrong"] | r["events"]["claimed_twice"]]

In [ ]:
r["rounds"].head(30)